# Circle detection from scratch: Circular Hough Transform vs RANSAC vs `cv2.HoughCircles`
Run the cells top to bottom. Tune Canny in Cell 1, set the diameter range in Cell 2, define the detectors in Cells 3-4, then use the **dashboard in Cell 5**: every slider re-runs all three methods on the same edge map and shows them side by side.

**Assumptions:** Canny edges use your from-scratch implementation by default (`cv2.Canny` is selectable for comparison); the image is processed at its original size (no downscaling), so all coordinates are image pixels.

## Cell 1 - Step A: load image (no resizing), tune Canny
Uses **your from-scratch Canny** by default (choose `cv2.Canny` in the *edge detector* dropdown to compare). Gaussian kernel size is a dropdown because your implementation only allows 3, 5, 7, 9, 15.

In [ ]:
# ============================================================
# CELL 1 - Step A: image loading, downscaling, interactive Canny tuning
# ============================================================
import io, math, json, time
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SEED = 0                # reproducibility (synthetic image noise + RANSAC)


def make_synthetic_image(w=480, h=360, seed=SEED):
    """Overlapping discs of different radii/intensities + Gaussian noise.
    Returns (uint8 gray image, ground-truth list of (x, y, r) in image pixels)."""
    rng = np.random.default_rng(seed)
    img = np.full((h, w), 60, np.uint8)
    gt = [(120, 130, 60, 200), (210, 150, 40, 130), (330, 120, 50, 235),
          (375, 185, 38, 160), (190, 265, 50, 110), (70, 285, 25, 220)]
    for (x, y, r, val) in gt:
        cv2.circle(img, (x, y), r, int(val), -1, lineType=cv2.LINE_AA)
    img = img.astype(np.float32) + rng.normal(0, 10, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8), [(x, y, r) for (x, y, r, _) in gt]


# ---- 1) image acquisition (upload -> fallback synthetic) ----
gray_full, GT_FULL, SOURCE = None, None, None
if IN_COLAB:
    print("Upload a .png/.jpg image (cancel / skip to use the synthetic test image) ...")
    try:
        uploaded = files.upload()
    except Exception as e:                       # user cancelled
        uploaded = {}
    for name, data in uploaded.items():
        if name.lower().endswith((".png", ".jpg", ".jpeg")):
            img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                gray_full, SOURCE = img, name
                break
        print(f"  skipped '{name}' (not a readable png/jpg)")

if gray_full is None:
    gray_full, GT_FULL = make_synthetic_image()
    SOURCE = "synthetic"
print(f"[A] source = {SOURCE}")

# ---- 2) grayscale, NO resizing: every coordinate is in original image pixels ----
gray_small = gray_full.copy()          # (name kept for the later cells; it is the full-size image)
f = 1.0                                # scale factor to original pixels (always 1: nothing is resized)
h, w = gray_small.shape
GT = list(GT_FULL) if GT_FULL else None
TICK = 20 if max(h, w) <= 800 else (50 if max(h, w) <= 2000 else 100)    # grid spacing readable at any size
mp = h * w / 1e6
print(f"[A] image size = {w}x{h} (WxH) = {mp:.2f} MP, processed at full size (no downscaling)")
if mp > 1.0:
    print(f"[A] note: {mp:.1f} MP is large - the pure-Python Canny needs roughly {1.33 * 5 * mp:.0f} s per run "
          f"(kernel 5) and the detectors get slower too; use the 'cv2.Canny' option while tuning if it is too slow.")


# ---- 3a) YOUR from-scratch Canny implementation (verbatim) ----
import numpy as np

ALLOWED_KERNEL_SIZES = (3, 5, 7, 9, 15)
SOBEL_X = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])
SOBEL_Y = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]])


def validate_parameters(kernel_size: int, sigma: float, high: float, low: float) -> None:
    if kernel_size not in ALLOWED_KERNEL_SIZES:
        raise ValueError(f"kernel_size must be one of {ALLOWED_KERNEL_SIZES}")
    if sigma <= 0:
        raise ValueError("sigma must be greater than zero")
    if high < low:
        raise ValueError("high_threshold must be greater than or equal to low_threshold")


def gaussian_kernel(image_dim: tuple[int, int], kernel_size: int, sigma: float) -> np.ndarray:
    height, width = image_dim
    if height < kernel_size or width < kernel_size:
        raise ValueError("The image is smaller than the selected kernel")
    if kernel_size not in ALLOWED_KERNEL_SIZES:
        raise ValueError(f"kernel_size must be one of {ALLOWED_KERNEL_SIZES}")
    center = kernel_size // 2
    x, y = np.meshgrid(
        np.linspace(-center, center, kernel_size),
        np.linspace(-center, center, kernel_size),
    )
    kernel = np.exp(-(x**2 + y**2) / (2 * sigma**2 + 1e-5))
    return kernel / np.sum(kernel)


def convolution(image: list[list[float]], kernel: list[list[float]]) -> list[list[float]]:
    height, width = len(image), len(image[0])
    kernel_height, kernel_width = len(kernel), len(kernel[0])
    output = [[0.0] * (width - kernel_width + 1) for _ in range(height - kernel_height + 1)]
    for i in range(len(output)):
        for j in range(len(output[0])):
            total = 0.0
            for u in range(kernel_height):
                for v in range(kernel_width):
                    total += image[i + u][j + v] * kernel[kernel_height - 1 - u][kernel_width - 1 - v]
            output[i][j] = total
    return output


def _pad(image: np.ndarray, kernel_size: int) -> np.ndarray:
    return np.pad(image.astype(np.float64), kernel_size // 2, mode="reflect")


def gaussian_blur(image: np.ndarray, kernel_size: int, sigma: float) -> np.ndarray:
    kernel = gaussian_kernel(image.shape, kernel_size, sigma)
    return np.array(convolution(_pad(image, kernel_size).tolist(), kernel.tolist()))


def apply_sobel(blurred: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    padded = _pad(blurred, 3)
    sobel_x = convolution(padded.tolist(), SOBEL_X.tolist())
    sobel_y = convolution(padded.tolist(), SOBEL_Y.tolist())
    x, y = np.array(sobel_x), np.array(sobel_y)
    return np.sqrt(x**2 + y**2), np.degrees(np.arctan2(y, x)) % 360


def non_maximum_suppression(gradient: np.ndarray, orientation: np.ndarray) -> np.ndarray:
    height, width = gradient.shape
    suppressed = np.zeros_like(gradient)
    for i in range(1, height - 1):
        for j in range(1, width - 1):
            angle = orientation[i, j] % 180
            if angle < 22.5 or angle >= 157.5:
                first, second = gradient[i, j - 1], gradient[i, j + 1]
            elif angle < 67.5:
                first, second = gradient[i - 1, j - 1], gradient[i + 1, j + 1]
            elif angle < 112.5:
                first, second = gradient[i - 1, j], gradient[i + 1, j]
            else:
                first, second = gradient[i - 1, j + 1], gradient[i + 1, j - 1]
            if gradient[i, j] >= max(first, second):
                suppressed[i, j] = gradient[i, j]
    return suppressed


def hysteresis_thresholding(image: np.ndarray, high: float, low: float) -> np.ndarray:
    result = np.zeros_like(image)
    stack: list[tuple[int, int]] = []
    strong = image >= high
    result[strong] = 255
    stack.extend(zip(*np.nonzero(strong)))
    while stack:
        row, column = stack.pop()
        for row_offset in (-1, 0, 1):
            for column_offset in (-1, 0, 1):
                r, c = row + row_offset, column + column_offset
                if 0 <= r < image.shape[0] and 0 <= c < image.shape[1]:
                    if result[r, c] == 0 and image[r, c] >= low:
                        result[r, c] = 255
                        stack.append((r, c))
    return result


def canny_edge_detector(
    image: np.ndarray, kernel_size: int, sigma: float, high_threshold: float, low_threshold: float
) -> dict[str, np.ndarray]:
    validate_parameters(kernel_size, sigma, high_threshold, low_threshold)
    grayscale = image if image.ndim == 2 else np.dot(image[..., :3], [0.299, 0.587, 0.114])
    blurred = gaussian_blur(grayscale, kernel_size, sigma)
    gradient, angles = apply_sobel(blurred)
    suppressed = non_maximum_suppression(gradient, angles)
    edges = hysteresis_thresholding(suppressed, high_threshold, low_threshold)
    return {
        "grayscale": grayscale,
        "gaussian_blur": blurred,
        "gradient_magnitude": gradient,
        "gradient_direction": angles,
        "non_maximum_suppression": suppressed,
        "edges": edges,
    }

# ---- 3b) Canny used by the notebook: YOUR implementation above (default) or cv2.Canny ----
CANNY_ENGINE = "scratch"      # "scratch" = your canny functions | "opencv" = cv2.Canny (faster; for comparison)
KERNEL_SIZE = 5               # Gaussian kernel of the scratch Canny: 3, 5, 7, 9 or 15
SIGMA, CANNY_LOW, CANNY_HIGH = 1.5, 30.0, 90.0     # values consumed by the next cells
edges, BLURRED_U8 = None, None                     # edge map + blurred image shared by ALL detectors
_CANNY_CACHE = {}
LAST_CANNY_INFO = ""


def scratch_canny_cached(gray, kernel_size, sigma, high, low):
    """Runs YOUR functions stage by stage (gaussian_blur -> apply_sobel -> non_maximum_suppression ->
    hysteresis_thresholding) and re-uses every stage whose inputs did not change, so moving only the
    thresholds is instant. Result is identical to canny_edge_detector(...)['edges']."""
    validate_parameters(kernel_size, sigma, high, low)
    c, ran = _CANNY_CACHE, []
    kb = (int(kernel_size), float(sigma))
    if c.get("kb") != kb:
        t0 = time.perf_counter(); c["blur"] = gaussian_blur(gray, kernel_size, sigma)
        c["grad"], c["ang"] = apply_sobel(c["blur"]); c["nms"] = non_maximum_suppression(c["grad"], c["ang"])
        c["kb"] = kb; c.pop("th", None)
        ran.append(f"blur+sobel+nms {time.perf_counter() - t0:.2f}s")
    th = (float(high), float(low))
    if c.get("th") != th:
        t0 = time.perf_counter()
        c["edges"] = (hysteresis_thresholding(c["nms"], high, low) > 0).astype(np.uint8) * 255
        c["th"] = th
        ran.append(f"hysteresis {time.perf_counter() - t0:.2f}s")
    return c["edges"], c["blur"], ", ".join(ran) if ran else "all stages cached"


def compute_edges(sigma, low, high):
    """Edge map + blurred image for the current CANNY_ENGINE (sets the globals BLURRED_U8, LAST_CANNY_INFO)."""
    global BLURRED_U8, LAST_CANNY_INFO
    if CANNY_ENGINE == "scratch":
        e, blur, info = scratch_canny_cached(gray_small, KERNEL_SIZE, float(sigma), float(high), float(low))
        BLURRED_U8 = np.clip(np.rint(blur), 0, 255).astype(np.uint8)
        LAST_CANNY_INFO = f"scratch Canny, kernel {KERNEL_SIZE}: {info}"
        return e
    BLURRED_U8 = cv2.GaussianBlur(gray_small, (0, 0), max(float(sigma), 0.1))
    LAST_CANNY_INFO = "cv2.Canny"
    return cv2.Canny(BLURRED_U8, float(low), float(high))


def preview_edges(sigma=SIGMA, low=CANNY_LOW, high=CANNY_HIGH, kernel_size=KERNEL_SIZE, engine=CANNY_ENGINE):
    """Left: grayscale + 20 px cyan grid (measure diameters). Right: Canny edge map."""
    global SIGMA, CANNY_LOW, CANNY_HIGH, KERNEL_SIZE, CANNY_ENGINE, edges
    if low > high:
        print(f"[A] low ({low}) > high ({high}) -> swapped")
        low, high = high, low
    SIGMA, CANNY_LOW, CANNY_HIGH = float(sigma), float(low), float(high)
    KERNEL_SIZE, CANNY_ENGINE = int(kernel_size), engine
    try:
        edges = compute_edges(SIGMA, CANNY_LOW, CANNY_HIGH)
    except ValueError as err:
        print("[A] Canny error:", err)
        return
    n_edge = int(np.count_nonzero(edges))

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    ax = axes[0]
    ax.imshow(gray_small, cmap="gray", vmin=0, vmax=255)
    ax.set_xticks(np.arange(0, w, TICK)); ax.set_yticks(np.arange(0, h, TICK))
    ax.tick_params(labelsize=7); plt.setp(ax.get_xticklabels(), rotation=90)
    ax.grid(True, color="cyan", linewidth=0.7, alpha=0.9)
    ax.set_title(f"Grayscale ({w}x{h}) - grid every {TICK} px: read diameters here")
    axes[1].imshow(edges, cmap="gray")
    axes[1].set_title(f"Canny [{CANNY_ENGINE}] | sigma={SIGMA:.1f}, low={CANNY_LOW:.0f}, high={CANNY_HIGH:.0f}"
                      f"{', kernel=' + str(KERNEL_SIZE) if CANNY_ENGINE == 'scratch' else ''} | {n_edge} edge px")
    axes[1].axis("off")
    plt.tight_layout(); plt.show()
    print(f"[A] edge pixels = {n_edge} | {LAST_CANNY_INFO}")


try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(preview_edges,
             sigma=FloatSlider(value=SIGMA, min=0.5, max=5.0, step=0.1, description="sigma", continuous_update=False),
             low=FloatSlider(value=CANNY_LOW, min=1, max=400, step=1, description="low", continuous_update=False),
             high=FloatSlider(value=CANNY_HIGH, min=1, max=500, step=1, description="high", continuous_update=False),
             kernel_size=Dropdown(options=list(ALLOWED_KERNEL_SIZES), value=KERNEL_SIZE, description="kernel"),
             engine=Dropdown(options=[("your Canny (scratch)", "scratch"), ("cv2.Canny", "opencv")],
                             value=CANNY_ENGINE, description="edge detector"))
    if edges is None:                       # widget did not fire (non-notebook environment)
        preview_edges()
except ImportError:
    print("[A] ipywidgets not available -> using default Canny parameters")
    preview_edges()

## Cell 2 - Step B: parameters and radius size guide

In [ ]:
SAMPLE_RATIO = 0.8   # fraction of a circle's perimeter we expect Canny to keep as edge pixels


def suggest_params(edges, min_diam, max_diam, min_votes=None, confidence=0.5, verbose=True):
    """Turn the user's intent (diameter range, sensitivity) into detector parameters."""
    warnings = []
    n_edge = int(np.count_nonzero(edges))
    if min_diam <= 0 or max_diam <= 0:
        raise ValueError("MIN_DIAM and MAX_DIAM must be > 0")
    if min_diam > max_diam:
        warnings.append(f"MIN_DIAM ({min_diam}) > MAX_DIAM ({max_diam}) -> values swapped")
        min_diam, max_diam = max_diam, min_diam
    confidence = float(np.clip(confidence, 0.1, 1.0))

    r_min, r_max = 0.85 * min_diam / 2.0, 1.15 * max_diam / 2.0
    perim_min = 2.0 * np.pi * r_min

    if min_votes is None:                                    # automatic
        cht_votes = max(5, int(confidence * perim_min * SAMPLE_RATIO))
        votes_src = "auto  = confidence * 2*pi*r_min * SAMPLE_RATIO"
    else:                                                    # user override is respected
        cht_votes = int(min_votes)
        votes_src = "user-defined MIN_VOTES"

    ransac_min_inliers = cht_votes                           # same evidence level for both methods
    perim_frac = confidence * SAMPLE_RATIO                   # supporters must also cover this share of 2*pi*r (CHT + RANSAC)
    min_cov = float(np.clip(0.2 + 0.8 * confidence, 0.25, 0.9))   # share of the 360 deg that must be supported (0.6 at default)
    tol = 1.5 if n_edge < 1000 else (2.0 if n_edge < 3000 else 2.5)
    max_iter = int(np.clip(1.5 * n_edge, 1000, 8000))

    if n_edge < 100:
        warnings.append(f"only {n_edge} edge px (< 100): lower Canny thresholds / sigma")
    clutter = max(5000, int(0.03 * edges.shape[0] * edges.shape[1]))     # 5000 px, or 3% of the image if larger
    if n_edge > clutter:
        warnings.append(f"{n_edge} edge px (> {clutter}): cluttered - raise Canny thresholds or sigma, "
                        f"expect false circles and slow RANSAC")

    P = dict(min_diam=min_diam, max_diam=max_diam, r_min=r_min, r_max=r_max, n_edge=n_edge,
             CHT_MIN_VOTES=cht_votes, MIN_COVERAGE=min_cov,
             RANSAC_MIN_INLIERS=ransac_min_inliers, PERIM_FRAC=perim_frac,
             RANSAC_INLIER_TOL=tol, RANSAC_MAX_ITER=max_iter, warnings=warnings)
    if verbose:
        print("--- suggest_params ---------------------------------------")
        print(f" edge pixels          : {n_edge}")
        print(f" diameter range       : {min_diam} .. {max_diam} px  ->  r_min={r_min:.1f}, r_max={r_max:.1f}")
        print(f" CHT_MIN_VOTES        : {cht_votes}   ({votes_src})")
        print(f" PERIM_FRAC           : {perim_frac:.2f}  (CHT + RANSAC need >= this share of 2*pi*r as support)")
        print(f" MIN_COVERAGE         : {min_cov:.2f}  (angular coverage of the 360 deg)")
        print(f" RANSAC_MIN_INLIERS   : {ransac_min_inliers}   (same as CHT_MIN_VOTES)")
        print(f" RANSAC_INLIER_TOL    : {tol} px")
        print(f" RANSAC_MAX_ITER      : {max_iter}  (per extraction round)")
        for msg in warnings:
            print(" WARNING:", msg)
        print("----------------------------------------------------------")
    return P

# ============================================================
# CELL 2 - Step B: user-controlled parameters + radius size guide
# ============================================================
# --- USER CONTROLS (image pixels: read them off the cyan grid of Cell 1) ---
MIN_DIAM = 45          # smallest circle diameter to detect
MAX_DIAM = 125         # largest circle diameter to detect
MIN_VOTES = None       # None = automatic (from MIN_CONFIDENCE) | or an integer, e.g. 60
MIN_CONFIDENCE = 0.5   # 0.1 (permissive) .. 1.0 (strict); used when MIN_VOTES is None
REF_CENTER = None      # (x, y) in processed px where the size guide is drawn, None = image centre

print("Guide: 1) measure the smallest/largest circle diameter on the Cell-1 grid.  "
      "2) Run this cell and check that the red (r_min) and yellow (r_max) circles bracket your objects.  "
      "3) Few detections -> lower MIN_CONFIDENCE (or MIN_VOTES); many false circles -> raise it.")

P = suggest_params(edges, MIN_DIAM, MAX_DIAM, MIN_VOTES, MIN_CONFIDENCE)
R_MIN, R_MAX = P["r_min"], P["r_max"]
ref = REF_CENTER if REF_CENTER is not None else (w / 2.0, h / 2.0)
print(f"[B] size guide centred at ({ref[0]:.0f}, {ref[1]:.0f}) | r_min={R_MIN:.1f}, r_max={R_MAX:.1f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, im, cmap, ttl in ((axes[0], gray_small, "gray", "Grayscale + radius guide"),
                          (axes[1], edges, "gray", f"Canny edges ({P['n_edge']} px) + radius guide")):
    ax.imshow(im, cmap=cmap)
    ax.add_patch(plt.Circle(ref, R_MIN, fill=False, color="red", ls="--", lw=2, label=f"r_min = {R_MIN:.1f}"))
    ax.add_patch(plt.Circle(ref, R_MAX, fill=False, color="yellow", ls="--", lw=2, label=f"r_max = {R_MAX:.1f}"))
    ax.set_title(ttl); ax.legend(loc="upper right", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## Cell 3 - Step C1: Circular Hough Transform from scratch
Definition only; it is run and visualised together with the other methods in the Cell 5 dashboard.

In [ ]:
# ============================================================
# CELL 3 - Step C1: Circular Hough Transform (CHT) from scratch
# ============================================================
# ---------- shared helpers (plain NumPy, no cv2 / scipy) ----------
def conv2d_same(img, kernel):
    """2-D correlation with edge padding, built from shifted array sums."""
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    H, W = img.shape
    p = np.pad(img.astype(np.float32), ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros((H, W), np.float32)
    for i in range(kh):
        for j in range(kw):
            if kernel[i, j] != 0:
                out += kernel[i, j] * p[i:i + H, j:j + W]
    return out


SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
SOBEL_Y = SOBEL_X.T.copy()


def sobel_gradients(gray):
    return conv2d_same(gray, SOBEL_X), conv2d_same(gray, SOBEL_Y)


def max_filter(img, size):
    """Sliding-window maximum (used for non-maximum suppression of accumulator peaks)."""
    r = size // 2
    H, W = img.shape
    p = np.pad(img, r, mode="constant", constant_values=-np.inf)
    out = np.full((H, W), -np.inf, np.float32)
    for i in range(size):
        for j in range(size):
            out = np.maximum(out, p[i:i + H, j:j + W])
    return out


def angular_coverage(dx, dy, n_bins=36):
    """Fraction of the 360 deg (in 10 deg bins) that has at least one supporting edge pixel."""
    if len(dx) == 0:
        return 0.0
    ang = np.arctan2(dy, dx)
    b = ((ang + np.pi) / (2 * np.pi) * n_bins).astype(int) % n_bins
    return np.unique(b).size / float(n_bins)


def fit_circle_lsq(x, y):
    """Algebraic (Kasa) least-squares circle fit: x^2+y^2 + D x + E y + F = 0."""
    A = np.column_stack([x, y, np.ones_like(x)])
    sol, *_ = np.linalg.lstsq(A, -(x ** 2 + y ** 2), rcond=None)
    a, b = -sol[0] / 2.0, -sol[1] / 2.0
    r2 = a * a + b * b - sol[2]
    return (a, b, float(np.sqrt(r2))) if r2 > 0 else None


def edge_support(edges, x, y, r, tol=1.5):
    """Number of edge pixels within +-tol px of the circle (same yardstick for every method)."""
    ys, xs = np.nonzero(edges)
    return int((np.abs(np.hypot(xs - x, ys - y) - r) < tol).sum())


def nms_circles(circles, center_frac=0.5, r_frac=0.25, min_center=4.0, min_dr=3.0):
    """Non-maximum suppression: drop a circle if a stronger one has (almost) the same
    centre AND (almost) the same radius. Concentric circles of different size are kept."""
    kept = []
    for c in sorted(circles, key=lambda c: -c["score"]):
        dup = False
        for k in kept:
            rm = min(c["r"], k["r"])
            if (np.hypot(c["x"] - k["x"], c["y"] - k["y"]) < max(min_center, center_frac * rm)
                    and abs(c["r"] - k["r"]) < max(min_dr, r_frac * rm)):
                dup = True
                break
        if not dup:
            kept.append(c)
    return kept


def circular_hough(gray_blur, edges, r_min, r_max, min_votes, min_coverage=0.5, perim_frac=0.0,
                   smooth=3, nms_win=7, align_cos=0.7, max_candidates=200, refine=True):
    """Circular Hough Transform from scratch (gradient-direction, 2-D centre accumulator).
    A circle needs max(min_votes, perim_frac * 2*pi*r) supporting edge pixels, so big circles need more votes.
    refine=True polishes each accepted circle with a least-squares fit on its supporting edge pixels (the accumulator
    centre can be several px off for large radii). Returns (circles, smoothed_accumulator);
    each circle = dict(x, y, r, score, coverage)."""
    H, W = edges.shape
    ys_i, xs_i = np.nonzero(edges)

    # 1) Sobel gradients -> unit normal vector at every edge pixel
    gx, gy = sobel_gradients(gray_blur.astype(np.float32))
    ex, ey = gx[ys_i, xs_i], gy[ys_i, xs_i]
    mag = np.hypot(ex, ey)
    ok = mag > 1e-6
    xs_i, ys_i, ex, ey, mag = xs_i[ok], ys_i[ok], ex[ok], ey[ok], mag[ok]
    if len(xs_i) < 3:
        return [], np.zeros((H, W), np.float32)
    nx, ny = ex / mag, ey / mag
    xs, ys = xs_i.astype(np.float64), ys_i.astype(np.float64)

    # 2) 2-D centre accumulator: vote along +-normal at every radius r_min..r_max
    acc = np.zeros(H * W, np.float64)
    radii = np.arange(int(np.floor(r_min)), int(np.ceil(r_max)) + 1)
    per_chunk = max(1, int(4_000_000 // (2 * len(xs))))     # radii per bincount call (keeps big images fast)
    for k in range(0, len(radii), per_chunk):
        votes = []
        for r in radii[k:k + per_chunk]:
            for s in (1.0, -1.0):                      # circle may be brighter or darker than background
                cx = np.rint(xs + s * r * nx).astype(np.int64)
                cy = np.rint(ys + s * r * ny).astype(np.int64)
                m = (cx >= 0) & (cx < W) & (cy >= 0) & (cy < H)
                votes.append(cy[m] * W + cx[m])
        acc += np.bincount(np.concatenate(votes), minlength=H * W)
    acc = acc.reshape(H, W).astype(np.float32)
    acc_s = conv2d_same(acc, np.ones((smooth, smooth), np.float32))   # absorb rounding errors

    # 3) local maxima of the accumulator = candidate centres
    is_peak = (acc_s >= max_filter(acc_s, nms_win)) & (acc_s >= 0.5 * min_votes)
    py, px = np.nonzero(is_peak)
    order = np.argsort(-acc_s[py, px])[:max_candidates]

    # 4) per candidate centre: radial-distance histogram -> supported radius,
    #    then perimeter support (vote count) + 360-degree angular coverage
    bins = np.arange(np.floor(r_min) - 0.5, np.ceil(r_max) + 1.5, 1.0)
    r_centers = 0.5 * (bins[:-1] + bins[1:])
    found = []
    for i in order:
        cx, cy = float(px[i]), float(py[i])
        dx, dy = xs - cx, ys - cy
        d = np.hypot(dx, dy)
        radial_ok = np.abs(dx * nx + dy * ny) / np.maximum(d, 1e-9) >= align_cos
        sel = (d >= r_min) & (d <= r_max) & radial_ok
        if sel.sum() < min_votes:
            continue
        ds = d[sel]
        hist, _ = np.histogram(ds, bins=bins)
        hs = np.convolve(hist, np.ones(3), mode="same")          # +-1 px smoothing
        for k in range(len(hs)):                                  # every radial peak (handles concentric circles)
            lo, hi = max(0, k - 2), min(len(hs), k + 3)
            if hs[k] < min_votes or hs[k] < hs[lo:hi].max() or (k > 0 and hs[k] == hs[k - 1]):
                continue
            r_est = float(ds[np.abs(ds - r_centers[k]) <= 1.5].mean())
            tol = 1.5 + 0.02 * r_est
            sup = sel & (np.abs(d - r_est) <= tol)
            ccx, ccy, rr = cx, cy, r_est
            if refine and sup.sum() >= 3:                       # least-squares polish of centre and radius
                for _ in range(2):
                    fit = fit_circle_lsq(xs[sup], ys[sup])
                    if fit is None or not (r_min <= fit[2] <= r_max):
                        break
                    ccx, ccy, rr = fit
                    dx, dy = xs - ccx, ys - ccy
                    d = np.hypot(dx, dy)
                    radial_ok = np.abs(dx * nx + dy * ny) / np.maximum(d, 1e-9) >= align_cos
                    sup = radial_ok & (np.abs(d - rr) <= 1.5 + 0.02 * rr)
            n_sup = int(sup.sum())
            cov = angular_coverage(dx[sup], dy[sup])
            if (n_sup >= max(min_votes, perim_frac * 2.0 * np.pi * rr)
                    and cov >= min_coverage and r_min <= rr <= r_max):
                found.append(dict(x=float(ccx), y=float(ccy), r=float(rr), score=n_sup, coverage=cov))

    # 5) NMS of duplicate detections
    return nms_circles(found), acc_s

print('[C1] CHT defined (see the Cell 5 dashboard)')

## Cell 4 - Step C2: RANSAC from scratch
Definition only; it is run and visualised in the Cell 5 dashboard.

In [ ]:
# ============================================================
# CELL 4 - Step C2: RANSAC circle detection from scratch
# ============================================================
def ransac_circles(edges, r_min, r_max, tol, max_iter, min_inliers, perim_frac=0.4,
                   min_coverage=0.5, batch=128, max_circles=30, refine=True, seed=0):
    """RANSAC circle detection from scratch.
    Each extraction round draws `max_iter` random 3-point samples, keeps the best circle,
    accepts it if it has enough support, removes its inliers and repeats."""
    rng = np.random.default_rng(seed)
    ys, xs = np.nonzero(edges)
    pts = np.column_stack([xs, ys]).astype(np.float64)
    circles = []

    for _ in range(max_circles):
        n = len(pts)
        if n < max(min_inliers, 3):
            break
        X, Y = pts[:, 0], pts[:, 1]
        best, drawn = None, 0
        while drawn < max_iter:
            B = min(batch, max_iter - drawn, max(8, int(4_000_000 // n)))     # cap memory for many edge points
            drawn += B
            idx = rng.integers(0, n, size=(B, 3))                         # 3-point sampling
            p1, p2, p3 = pts[idx[:, 0]], pts[idx[:, 1]], pts[idx[:, 2]]
            # circumcentre = intersection of two perpendicular bisectors:
            #   2(p2-p1).c = |p2|^2-|p1|^2   and   2(p3-p1).c = |p3|^2-|p1|^2
            a11, a12 = 2 * (p2[:, 0] - p1[:, 0]), 2 * (p2[:, 1] - p1[:, 1])
            a21, a22 = 2 * (p3[:, 0] - p1[:, 0]), 2 * (p3[:, 1] - p1[:, 1])
            b1 = (p2 ** 2).sum(1) - (p1 ** 2).sum(1)
            b2 = (p3 ** 2).sum(1) - (p1 ** 2).sum(1)
            det = a11 * a22 - a12 * a21
            ok = (np.abs(det) > 8.0) & (idx[:, 0] != idx[:, 1]) & (idx[:, 0] != idx[:, 2]) & (idx[:, 1] != idx[:, 2])
            det = np.where(ok, det, 1.0)                                  # non-collinear only
            ca = (b1 * a22 - a12 * b2) / det
            cb = (a11 * b2 - b1 * a21) / det
            cr = np.hypot(p1[:, 0] - ca, p1[:, 1] - cb)
            ok &= (cr >= r_min) & (cr <= r_max)                           # radius filter
            if not ok.any():
                continue
            ca, cb, cr = ca[ok], cb[ok], cr[ok]
            dist = np.abs(np.hypot(X[None, :] - ca[:, None], Y[None, :] - cb[:, None]) - cr[:, None])
            cnt = (dist < tol).sum(1)                                     # inlier scoring
            j = int(np.argmax(cnt))
            if best is None or cnt[j] > best[0]:
                best = (int(cnt[j]), ca[j], cb[j], cr[j])
        if best is None:
            break

        _, a, b, r = best
        mask = np.abs(np.hypot(X - a, Y - b) - r) < tol
        if refine and mask.sum() >= 3:                                    # least-squares polish on all inliers
            fit = fit_circle_lsq(X[mask], Y[mask])
            if fit is not None and r_min <= fit[2] <= r_max:
                a, b, r = fit
                mask = np.abs(np.hypot(X - a, Y - b) - r) < tol
        n_in = int(mask.sum())
        cov = angular_coverage(X[mask] - a, Y[mask] - b)
        if n_in < max(min_inliers, perim_frac * 2 * np.pi * r) or cov < min_coverage:
            break                                                         # best of this round is not a circle -> done
        circles.append(dict(x=float(a), y=float(b), r=float(r), score=n_in, coverage=float(cov)))
        pts = pts[~mask]                                                  # remove inliers, extract next circle
    return circles

print('[C2] RANSAC defined (see the Cell 5 dashboard)')

## Cell 5 - Step D: OpenCV baseline + interactive dashboard (all three methods at once)
The tabs hold the shared edge/size controls and the CHT, RANSAC and OpenCV parameters. Moving any slider updates all three panels; only the method whose inputs changed is recomputed.

In [ ]:
# ============================================================
# CELL 5 - Step D: OpenCV baseline + one dashboard for all three methods
# ============================================================
def detect_circles_opencv(gray_small, r_min, r_max, min_dist, param1=100, param2=30, sigma=1.5, dp=1.0,
                          pre_blurred=False):
    """cv2.HoughCircles (HOUGH_GRADIENT). param1 = upper Canny threshold, param2 = accumulator threshold.
    pre_blurred=True: gray_small is already the blurred image shared with the other two methods."""
    blur = gray_small if pre_blurred else cv2.GaussianBlur(gray_small, (0, 0), sigma)
    out = cv2.HoughCircles(blur, cv2.HOUGH_GRADIENT, dp=float(dp), minDist=float(min_dist),
                           param1=float(param1), param2=float(param2),
                           minRadius=int(np.floor(r_min)), maxRadius=int(np.ceil(r_max)))
    if out is None:
        return []
    return [dict(x=float(x), y=float(y), r=float(r)) for x, y, r in out[0]]


def timed(fn, repeats=3, max_total_ms=2500.0):
    """Run fn() up to `repeats` times (fewer if it is slow); return (last result, median ms)."""
    times, res = [], None
    for _ in range(repeats):
        t0 = time.perf_counter()
        res = fn()
        times.append((time.perf_counter() - t0) * 1000.0)
        if sum(times) > max_total_ms:
            break
    return res, float(np.median(times))


# ---------- helpers for the interactive dashboard ----------
try:
    import ipywidgets as W
    from IPython.display import display
    HAVE_W = True
except ImportError:
    HAVE_W = False

RESULTS = {}          # latest detections of the three methods, read by Cell 6


def sl_int(v, lo, hi, step, desc):
    return W.IntSlider(value=int(min(max(v, lo), hi)), min=lo, max=hi, step=step, description=desc,
                       continuous_update=False, style={"description_width": "170px"},
                       layout=W.Layout(width="600px"))


def sl_float(v, lo, hi, step, desc):
    return W.FloatSlider(value=float(min(max(v, lo), hi)), min=lo, max=hi, step=step, description=desc,
                         continuous_update=False, style={"description_width": "170px"},
                         layout=W.Layout(width="600px"), readout_format=".2f")


def draw_circles(ax, circles, color, lw=2.2, numbered=True):
    for i, c in enumerate(sorted(circles, key=lambda c: -c["r"]), 1):
        ax.add_patch(plt.Circle((c["x"], c["y"]), c["r"], fill=False, color=color, lw=lw))
        ax.plot(c["x"], c["y"], marker="+", color=color, ms=9, mew=1.5)
        if numbered:
            ax.text(c["x"] + 3, c["y"] - 3, str(i), color="yellow", fontsize=9, weight="bold")


def diam_to_radii(min_diam, max_diam):
    if min_diam > max_diam:
        min_diam, max_diam = max_diam, min_diam
    return 0.85 * min_diam / 2.0, 1.15 * max_diam / 2.0


DIAM_HI = int(max(w, h))


# ============================================================
# ONE dashboard: every slider re-runs CHT + RANSAC + OpenCV on the SAME edge map
# ============================================================
SHARED = dict(min_diam=MIN_DIAM, max_diam=MAX_DIAM, sigma=SIGMA, high=CANNY_HIGH, low=CANNY_LOW, match=False,
              kernel=KERNEL_SIZE, engine=CANNY_ENGINE)
CHT_PARAMS = dict(min_votes=P["CHT_MIN_VOTES"], min_coverage=P["MIN_COVERAGE"], perim_frac=P["PERIM_FRAC"],
                  smooth=3, nms_win=7, align_cos=0.7, refine=True)
RANSAC_PARAMS = dict(tol=P["RANSAC_INLIER_TOL"], max_iter=P["RANSAC_MAX_ITER"], min_inliers=P["RANSAC_MIN_INLIERS"],
                     perim_frac=P["PERIM_FRAC"], min_coverage=P["MIN_COVERAGE"], refine=True, seed=SEED)
OPENCV_PARAMS = dict(dp=1.0, min_dist=max(5.0, R_MIN), param2=max(10, int(0.6 * P["CHT_MIN_VOTES"])))
USE = dict(cht=True, ransac=True, opencv=True)
VOTE_MAX = max(800, int(np.pi * DIAM_HI))      # slider range for vote / inlier thresholds (a circle can have ~pi*size edge px)
_CACHE = {}


def current_radii():
    return diam_to_radii(SHARED["min_diam"], SHARED["max_diam"])


def update_edges():
    """ONE edge map + ONE blurred image shared by CHT, RANSAC and OpenCV. With 'match' on, low = high/2
    (what cv2.HoughCircles uses internally; it only exposes the high threshold, param1)."""
    global edges, SIGMA, CANNY_LOW, CANNY_HIGH, KERNEL_SIZE, CANNY_ENGINE
    SIGMA, CANNY_HIGH = float(SHARED["sigma"]), float(SHARED["high"])
    KERNEL_SIZE, CANNY_ENGINE = int(SHARED["kernel"]), SHARED["engine"]
    CANNY_LOW = max(CANNY_HIGH / 2.0, 1.0) if SHARED["match"] else min(float(SHARED["low"]), CANNY_HIGH)
    edges = compute_edges(SIGMA, CANNY_LOW, CANNY_HIGH)


def detect_cht():
    p, (r_min, r_max) = CHT_PARAMS, current_radii()
    return circular_hough(BLURRED_U8, edges, r_min, r_max, int(p["min_votes"]), float(p["min_coverage"]),
                          float(p["perim_frac"]), int(p["smooth"]) | 1, int(p["nms_win"]) | 1, float(p["align_cos"]),
                          refine=bool(p["refine"]))


def detect_ransac():
    p, (r_min, r_max) = RANSAC_PARAMS, current_radii()
    return ransac_circles(edges, r_min, r_max, float(p["tol"]), int(p["max_iter"]), int(p["min_inliers"]),
                          float(p["perim_frac"]), float(p["min_coverage"]), refine=bool(p["refine"]),
                          seed=int(p["seed"]))


def detect_opencv():
    p, (r_min, r_max) = OPENCV_PARAMS, current_radii()
    return detect_circles_opencv(BLURRED_U8, r_min, r_max, p["min_dist"], CANNY_HIGH, p["param2"], SIGMA,
                                 p["dp"], pre_blurred=True)


def _scored(circles):
    for c in circles:                       # one common yardstick: edge pixels within +-1.5 px
        c["score"] = c.get("score", edge_support(edges, c["x"], c["y"], c["r"]))
    return circles


def run_timing(repeats=3, verbose=True):
    """Re-time the three methods with the current slider values (median of up to 3 runs)."""
    for name, fn in (("CHT (scratch)", lambda: detect_cht()[0]), ("RANSAC (scratch)", detect_ransac),
                     ("OpenCV HoughCircles", detect_opencv)):
        circles, ms = timed(fn, repeats)
        RESULTS[name] = dict(circles=_scored(circles), ms=ms)
        if verbose:
            print(f"[D] {name:<20s}: {len(circles):2d} circles | {ms:8.1f} ms")
    if verbose:
        print("[D] note: CHT/RANSAC times exclude Canny; CHT includes blur + Sobel; "
              "OpenCV's time includes its internal Canny.")


def dashboard(**kw):
    SHARED.update(min_diam=kw["min_diam"], max_diam=kw["max_diam"], sigma=kw["sigma"], high=kw["high"],
                  low=kw["low"], match=kw["match"], kernel=kw["kernel"], engine=kw["engine"])
    CHT_PARAMS.update(min_votes=kw["cht_votes"], min_coverage=kw["cht_cov"], perim_frac=kw["cht_perim"],
                      smooth=kw["cht_smooth"], nms_win=kw["cht_nms"], align_cos=kw["cht_align"], refine=kw["cht_refine"])
    RANSAC_PARAMS.update(tol=kw["ran_tol"], max_iter=kw["ran_iter"], min_inliers=kw["ran_min_in"],
                         perim_frac=kw["ran_perim"], min_coverage=kw["ran_cov"], refine=kw["ran_refine"],
                         seed=kw["ran_seed"])
    OPENCV_PARAMS.update(dp=kw["cv_dp"], min_dist=kw["cv_min_dist"], param2=kw["cv_param2"])
    USE.update(cht=kw["cht_on"], ransac=kw["ran_on"], opencv=kw["cv_on"])
    update_edges()
    r_min, r_max = current_radii()
    ekey = (CANNY_ENGINE, KERNEL_SIZE, SIGMA, CANNY_HIGH, CANNY_LOW, SHARED["min_diam"], SHARED["max_diam"])

    def run(name, on, params, fn):                  # only re-run a method if ITS inputs changed
        if not on:
            return None
        key = (ekey, tuple(sorted(params.items())))
        hit = _CACHE.get(name)
        if hit is not None and hit[0] == key:
            return hit[1]
        t0 = time.perf_counter()
        res = fn()
        _CACHE[name] = (key, (res, (time.perf_counter() - t0) * 1000.0))
        return _CACHE[name][1]

    cht = run("cht", USE["cht"], CHT_PARAMS, detect_cht)
    ran = run("ransac", USE["ransac"], RANSAC_PARAMS, detect_ransac)
    ocv = run("opencv", USE["opencv"], OPENCV_PARAMS, detect_opencv)
    panels = [("CHT (scratch)", "red", None if cht is None else (cht[0][0], cht[1])),
              ("RANSAC (scratch)", "deepskyblue", ran),
              ("OpenCV HoughCircles", "lime", ocv)]
    for name, _, res in panels:
        if res is not None:
            RESULTS[name] = dict(circles=_scored(res[0]), ms=res[1])

    # does OpenCV's internal Canny (low = param1/2, high = param1) produce the same pixels as our edge map?
    cv_edges = cv2.Canny(BLURRED_U8, max(CANNY_HIGH / 2.0, 1.0), CANNY_HIGH)
    both = int(np.count_nonzero((cv_edges > 0) & (edges > 0)))
    union = max(int(np.count_nonzero((cv_edges > 0) | (edges > 0))), 1)
    same = "IDENTICAL" if np.array_equal(cv_edges > 0, edges > 0) else f"{100.0 * both / union:.0f}% overlap"

    fig, axes = plt.subplots(2, 2, figsize=(13, 10.8))
    ax = axes[0, 0]
    ax.imshow(edges, cmap="gray")
    ax.add_patch(plt.Circle(ref, r_min, fill=False, color="red", ls="--", lw=1.8))
    ax.add_patch(plt.Circle(ref, r_max, fill=False, color="yellow", ls="--", lw=1.8))
    ax.set_title(f"SHARED edge map [{CANNY_ENGINE}]: {int(np.count_nonzero(edges))} px | "
                 f"sigma={SIGMA:.1f}, low={CANNY_LOW:.0f}, high={CANNY_HIGH:.0f}"
                 f"{', kernel=' + str(KERNEL_SIZE) if CANNY_ENGINE == 'scratch' else ''}\n"
                 f"r {r_min:.0f}-{r_max:.0f} | OpenCV's own internal edges vs shared: {same}", fontsize=9)
    ax.axis("off")
    for ax, (name, col, res) in zip((axes[0, 1], axes[1, 0], axes[1, 1]), panels):
        ax.imshow(gray_small, cmap="gray", vmin=0, vmax=255)
        if res is None:
            ax.set_title(f"{name}: switched off"); ax.axis("off"); continue
        draw_circles(ax, res[0], col)
        ax.set_title(f"{name}: {len(res[0])} circles | {res[1]:.0f} ms"); ax.axis("off")
    plt.tight_layout(); plt.show()
    print("   ".join(f"{n.split(' ')[0]}: {len(r[0])} circles, {r[1]:.0f} ms" if r else f"{n.split(' ')[0]}: off"
                     for n, _, r in panels))


def _controls():
    c = {}
    c["min_diam"] = sl_int(SHARED["min_diam"], 4, DIAM_HI, 1, "min diameter")
    c["max_diam"] = sl_int(SHARED["max_diam"], 4, DIAM_HI, 1, "max diameter")
    c["engine"] = W.Dropdown(options=[("your Canny (scratch)", "scratch"), ("cv2.Canny", "opencv")],
                             value=SHARED["engine"], description="edge detector",
                             style={"description_width": "170px"}, layout=W.Layout(width="600px"))
    c["kernel"] = W.Dropdown(options=list(ALLOWED_KERNEL_SIZES), value=SHARED["kernel"],
                             description="Gaussian kernel (scratch)", style={"description_width": "170px"},
                             layout=W.Layout(width="600px"))
    c["sigma"] = sl_float(SHARED["sigma"], 0.5, 5.0, 0.1, "Canny sigma")
    c["high"] = sl_float(SHARED["high"], 5, 500, 1, "Canny high")
    c["low"] = sl_float(SHARED["low"], 1, 400, 1, "Canny low")
    c["match"] = W.Checkbox(value=SHARED["match"], indent=False, layout=W.Layout(width="600px"),
                            description="low = high/2  (what OpenCV uses internally; exact match with cv2.Canny)")
    c["low"].disabled = c["match"].value
    c["match"].observe(lambda ch: setattr(c["low"], "disabled", ch["new"]), names="value")
    c["cht_on"] = W.Checkbox(value=True, description="run CHT", indent=False)
    c["cht_votes"] = sl_int(CHT_PARAMS["min_votes"], 5, VOTE_MAX, 1, "min votes")
    c["cht_perim"] = sl_float(CHT_PARAMS["perim_frac"], 0.0, 1.0, 0.05, "min perimeter share")
    c["cht_cov"] = sl_float(CHT_PARAMS["min_coverage"], 0.1, 1.0, 0.05, "min coverage (360)")
    c["cht_smooth"] = sl_int(CHT_PARAMS["smooth"], 1, 9, 2, "accumulator smooth")
    c["cht_nms"] = sl_int(CHT_PARAMS["nms_win"], 3, 21, 2, "peak NMS window")
    c["cht_align"] = sl_float(CHT_PARAMS["align_cos"], 0.0, 0.95, 0.05, "gradient alignment")
    c["cht_refine"] = W.Checkbox(value=CHT_PARAMS["refine"], description="least-squares refine (centre/radius)", indent=False)
    c["ran_on"] = W.Checkbox(value=True, description="run RANSAC", indent=False)
    c["ran_tol"] = sl_float(RANSAC_PARAMS["tol"], 0.5, 6.0, 0.1, "inlier tolerance (px)")
    c["ran_iter"] = sl_int(RANSAC_PARAMS["max_iter"], 100, 10000, 100, "iterations / round")
    c["ran_min_in"] = sl_int(RANSAC_PARAMS["min_inliers"], 5, VOTE_MAX, 1, "min inliers")
    c["ran_perim"] = sl_float(RANSAC_PARAMS["perim_frac"], 0.0, 1.0, 0.05, "min perimeter share")
    c["ran_cov"] = sl_float(RANSAC_PARAMS["min_coverage"], 0.1, 1.0, 0.05, "min coverage (360)")
    c["ran_refine"] = W.Checkbox(value=RANSAC_PARAMS["refine"], description="least-squares refine", indent=False)
    c["ran_seed"] = sl_int(RANSAC_PARAMS["seed"], 0, 99, 1, "random seed")
    c["cv_on"] = W.Checkbox(value=True, description="run OpenCV", indent=False)
    c["cv_dp"] = sl_float(OPENCV_PARAMS["dp"], 0.5, 3.0, 0.1, "dp (accum. resolution)")
    c["cv_min_dist"] = sl_float(OPENCV_PARAMS["min_dist"], 1, max(200, DIAM_HI // 2), 1, "min centre distance")
    c["cv_param2"] = sl_float(OPENCV_PARAMS["param2"], 5, VOTE_MAX, 1, "param2 (accum. threshold)")
    return c


DASH_VALUES = dict(min_diam=SHARED["min_diam"], max_diam=SHARED["max_diam"], sigma=SHARED["sigma"],
                   high=SHARED["high"], low=SHARED["low"], match=False,
                   kernel=SHARED["kernel"], engine=SHARED["engine"], cht_on=True, cht_votes=CHT_PARAMS["min_votes"], cht_perim=CHT_PARAMS["perim_frac"],
                   cht_cov=CHT_PARAMS["min_coverage"], cht_smooth=3, cht_nms=7, cht_align=0.7, cht_refine=True,
                   ran_on=True, ran_tol=RANSAC_PARAMS["tol"], ran_iter=RANSAC_PARAMS["max_iter"],
                   ran_min_in=RANSAC_PARAMS["min_inliers"], ran_perim=RANSAC_PARAMS["perim_frac"],
                   ran_cov=RANSAC_PARAMS["min_coverage"], ran_refine=True, ran_seed=SEED,
                   cv_on=True, cv_dp=1.0, cv_min_dist=OPENCV_PARAMS["min_dist"], cv_param2=OPENCV_PARAMS["param2"])

if HAVE_W:
    ctrl = _controls()
    tab = W.Tab(children=[
        W.VBox([ctrl[k] for k in ("min_diam", "max_diam", "engine", "kernel", "sigma", "high", "low", "match")]),
        W.VBox([ctrl[k] for k in ("cht_on", "cht_votes", "cht_perim", "cht_cov", "cht_smooth", "cht_nms", "cht_align", "cht_refine")]),
        W.VBox([ctrl[k] for k in ("ran_on", "ran_tol", "ran_iter", "ran_min_in", "ran_perim", "ran_cov", "ran_refine", "ran_seed")]),
        W.VBox([ctrl[k] for k in ("cv_on", "cv_dp", "cv_min_dist", "cv_param2")])])
    for i, t in enumerate(("Shared: edges + size", "CHT", "RANSAC", "OpenCV")):
        tab.set_title(i, t)
    display(tab, W.interactive_output(dashboard, ctrl))
    if not RESULTS:                                  # widget did not fire (non-notebook environment)
        dashboard(**DASH_VALUES)
else:
    print("[ipywidgets not available -> running once with the default parameters]")
    dashboard(**DASH_VALUES)

print("\n=== Timing with the current slider values ===")
run_timing()

## Cell 6 - Step E: benchmark
Re-times the three methods with the current dashboard values, so re-run this cell after tuning.

In [ ]:
# ============================================================
# CELL 6 - Step E: visual + quantitative benchmarking
# ============================================================
print("Re-timing the three methods with the current dashboard values ...")
run_timing()
RN, RX = current_radii()

STYLE = {"CHT (scratch)": ("red", "Custom CHT (from scratch)"),
         "RANSAC (scratch)": ("deepskyblue", "Custom RANSAC (from scratch)"),
         "OpenCV HoughCircles": ("lime", "OpenCV cv2.HoughCircles")}

fig, axes = plt.subplots(1, 5, figsize=(27, 5.6))

# 1) grayscale + centre crosshairs (big white = REF_CENTER, small coloured = detected centres)
ax = axes[0]
ax.imshow(gray_small, cmap="gray", vmin=0, vmax=255)
ax.plot(*ref, marker="+", color="white", ms=22, mew=2)
for name, (col, _) in STYLE.items():
    for c in RESULTS[name]["circles"]:
        ax.plot(c["x"], c["y"], marker="+", color=col, ms=12, mew=2)
ax.set_title("Grayscale + centre crosshairs"); ax.axis("off")

# 2) edge map + radius size guide
ax = axes[1]
ax.imshow(edges, cmap="gray")
ax.add_patch(plt.Circle(ref, RN, fill=False, color="red", ls="--", lw=2))
ax.add_patch(plt.Circle(ref, RX, fill=False, color="yellow", ls="--", lw=2))
ax.set_title(f"Canny edges + size guide (r {RN:.0f}-{RX:.0f})"); ax.axis("off")

# 3-5) detections of each method
for ax, (name, (col, ttl)) in zip(axes[2:], STYLE.items()):
    ax.imshow(gray_small, cmap="gray", vmin=0, vmax=255)
    circs = RESULTS[name]["circles"]
    for c in circs:
        ax.add_patch(plt.Circle((c["x"], c["y"]), c["r"], fill=False, color=col, lw=2.2))
        ax.plot(c["x"], c["y"], marker="+", color=col, ms=9, mew=1.5)
    ax.set_title(f"{ttl}\n{len(circs)} circles | {RESULTS[name]['ms']:.1f} ms"); ax.axis("off")
plt.tight_layout(); plt.show()

# ---- ASCII comparison table (processed-pixel coordinates) ----
line = "+" + "+".join("-" * n for n in (22, 4, 17, 9, 14, 11)) + "+"
print(line)
print(f"| {'Method':<20s} | {'ID':>2s} | {'Center (x, y)':<15s} | {'Radius':>7s} | {'Support':<12s} | {'Time (ms)':>9s} |")
print(line)
for name in STYLE:
    circs, ms = RESULTS[name]["circles"], RESULTS[name]["ms"]
    if not circs:
        print(f"| {name:<20s} | {'-':>2s} | {'(none)':<15s} | {'-':>7s} | {'-':<12s} | {ms:9.1f} |")
    for i, c in enumerate(sorted(circs, key=lambda c: -c["r"]), 1):
        pct = 100.0 * c["score"] / (2 * np.pi * c["r"])
        t = f"{ms:9.1f}" if i == 1 else " " * 9
        print(f"| {name:<20s} | {i:>2d} | ({c['x']:6.1f},{c['y']:6.1f}) | {c['r']:7.2f} | {c['score']:>5d} ({pct:3.0f}%) | {t} |")
    print(line)
print("Support = edge pixels within +-1.5 px of the circle (same yardstick for all methods; % of circumference).")

# ---- accuracy vs. ground truth (only available for the synthetic image) ----
if GT:
    def match(dets, gts):
        used, tp, ce, re_ = set(), 0, [], []
        for (gx_, gy_, gr_) in gts:
            best = None
            for j, d in enumerate(dets):
                if j in used:
                    continue
                dc = np.hypot(d["x"] - gx_, d["y"] - gy_)
                dr = abs(d["r"] - gr_)
                if dc <= max(4, 0.3 * gr_) and dr <= max(3, 0.25 * gr_) and (best is None or dc < best[1]):
                    best = (j, dc, dr)
            if best:
                used.add(best[0]); tp += 1; ce.append(best[1]); re_.append(best[2])
        return tp, len(dets) - tp, len(gts) - tp, ce, re_

    print("\nAccuracy vs ground truth (synthetic image, {} true circles)".format(len(GT)))
    print(f"{'Method':<22s}{'TP':>4s}{'FP':>4s}{'FN':>4s}{'mean centre err':>17s}{'mean |dr|':>11s}")
    for name in STYLE:
        tp, fp, fn, ce, re_ = match(RESULTS[name]["circles"], GT)
        print(f"{name:<22s}{tp:>4d}{fp:>4d}{fn:>4d}{(np.mean(ce) if ce else float('nan')):>14.2f} px{(np.mean(re_) if re_ else float('nan')):>8.2f} px")
else:
    print("\n(no ground truth for uploaded images - judge accuracy visually)")

## Cell 7 - Step F: exportable pipeline

In [ ]:
# ============================================================
# CELL 7 - Step F: backend-ready, self-contained module
# (imports + helpers + detectors + pipeline; uses no notebook globals)
# ============================================================
import time, json
import numpy as np
import cv2

import numpy as np

ALLOWED_KERNEL_SIZES = (3, 5, 7, 9, 15)
SOBEL_X = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])
SOBEL_Y = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]])


def validate_parameters(kernel_size: int, sigma: float, high: float, low: float) -> None:
    if kernel_size not in ALLOWED_KERNEL_SIZES:
        raise ValueError(f"kernel_size must be one of {ALLOWED_KERNEL_SIZES}")
    if sigma <= 0:
        raise ValueError("sigma must be greater than zero")
    if high < low:
        raise ValueError("high_threshold must be greater than or equal to low_threshold")


def gaussian_kernel(image_dim: tuple[int, int], kernel_size: int, sigma: float) -> np.ndarray:
    height, width = image_dim
    if height < kernel_size or width < kernel_size:
        raise ValueError("The image is smaller than the selected kernel")
    if kernel_size not in ALLOWED_KERNEL_SIZES:
        raise ValueError(f"kernel_size must be one of {ALLOWED_KERNEL_SIZES}")
    center = kernel_size // 2
    x, y = np.meshgrid(
        np.linspace(-center, center, kernel_size),
        np.linspace(-center, center, kernel_size),
    )
    kernel = np.exp(-(x**2 + y**2) / (2 * sigma**2 + 1e-5))
    return kernel / np.sum(kernel)


def convolution(image: list[list[float]], kernel: list[list[float]]) -> list[list[float]]:
    height, width = len(image), len(image[0])
    kernel_height, kernel_width = len(kernel), len(kernel[0])
    output = [[0.0] * (width - kernel_width + 1) for _ in range(height - kernel_height + 1)]
    for i in range(len(output)):
        for j in range(len(output[0])):
            total = 0.0
            for u in range(kernel_height):
                for v in range(kernel_width):
                    total += image[i + u][j + v] * kernel[kernel_height - 1 - u][kernel_width - 1 - v]
            output[i][j] = total
    return output


def _pad(image: np.ndarray, kernel_size: int) -> np.ndarray:
    return np.pad(image.astype(np.float64), kernel_size // 2, mode="reflect")


def gaussian_blur(image: np.ndarray, kernel_size: int, sigma: float) -> np.ndarray:
    kernel = gaussian_kernel(image.shape, kernel_size, sigma)
    return np.array(convolution(_pad(image, kernel_size).tolist(), kernel.tolist()))


def apply_sobel(blurred: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    padded = _pad(blurred, 3)
    sobel_x = convolution(padded.tolist(), SOBEL_X.tolist())
    sobel_y = convolution(padded.tolist(), SOBEL_Y.tolist())
    x, y = np.array(sobel_x), np.array(sobel_y)
    return np.sqrt(x**2 + y**2), np.degrees(np.arctan2(y, x)) % 360


def non_maximum_suppression(gradient: np.ndarray, orientation: np.ndarray) -> np.ndarray:
    height, width = gradient.shape
    suppressed = np.zeros_like(gradient)
    for i in range(1, height - 1):
        for j in range(1, width - 1):
            angle = orientation[i, j] % 180
            if angle < 22.5 or angle >= 157.5:
                first, second = gradient[i, j - 1], gradient[i, j + 1]
            elif angle < 67.5:
                first, second = gradient[i - 1, j - 1], gradient[i + 1, j + 1]
            elif angle < 112.5:
                first, second = gradient[i - 1, j], gradient[i + 1, j]
            else:
                first, second = gradient[i - 1, j + 1], gradient[i + 1, j - 1]
            if gradient[i, j] >= max(first, second):
                suppressed[i, j] = gradient[i, j]
    return suppressed


def hysteresis_thresholding(image: np.ndarray, high: float, low: float) -> np.ndarray:
    result = np.zeros_like(image)
    stack: list[tuple[int, int]] = []
    strong = image >= high
    result[strong] = 255
    stack.extend(zip(*np.nonzero(strong)))
    while stack:
        row, column = stack.pop()
        for row_offset in (-1, 0, 1):
            for column_offset in (-1, 0, 1):
                r, c = row + row_offset, column + column_offset
                if 0 <= r < image.shape[0] and 0 <= c < image.shape[1]:
                    if result[r, c] == 0 and image[r, c] >= low:
                        result[r, c] = 255
                        stack.append((r, c))
    return result


def canny_edge_detector(
    image: np.ndarray, kernel_size: int, sigma: float, high_threshold: float, low_threshold: float
) -> dict[str, np.ndarray]:
    validate_parameters(kernel_size, sigma, high_threshold, low_threshold)
    grayscale = image if image.ndim == 2 else np.dot(image[..., :3], [0.299, 0.587, 0.114])
    blurred = gaussian_blur(grayscale, kernel_size, sigma)
    gradient, angles = apply_sobel(blurred)
    suppressed = non_maximum_suppression(gradient, angles)
    edges = hysteresis_thresholding(suppressed, high_threshold, low_threshold)
    return {
        "grayscale": grayscale,
        "gaussian_blur": blurred,
        "gradient_magnitude": gradient,
        "gradient_direction": angles,
        "non_maximum_suppression": suppressed,
        "edges": edges,
    }


# ---------- shared helpers (plain NumPy, no cv2 / scipy) ----------
def conv2d_same(img, kernel):
    """2-D correlation with edge padding, built from shifted array sums."""
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    H, W = img.shape
    p = np.pad(img.astype(np.float32), ((ph, ph), (pw, pw)), mode="edge")
    out = np.zeros((H, W), np.float32)
    for i in range(kh):
        for j in range(kw):
            if kernel[i, j] != 0:
                out += kernel[i, j] * p[i:i + H, j:j + W]
    return out


SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
SOBEL_Y = SOBEL_X.T.copy()


def sobel_gradients(gray):
    return conv2d_same(gray, SOBEL_X), conv2d_same(gray, SOBEL_Y)


def max_filter(img, size):
    """Sliding-window maximum (used for non-maximum suppression of accumulator peaks)."""
    r = size // 2
    H, W = img.shape
    p = np.pad(img, r, mode="constant", constant_values=-np.inf)
    out = np.full((H, W), -np.inf, np.float32)
    for i in range(size):
        for j in range(size):
            out = np.maximum(out, p[i:i + H, j:j + W])
    return out


def angular_coverage(dx, dy, n_bins=36):
    """Fraction of the 360 deg (in 10 deg bins) that has at least one supporting edge pixel."""
    if len(dx) == 0:
        return 0.0
    ang = np.arctan2(dy, dx)
    b = ((ang + np.pi) / (2 * np.pi) * n_bins).astype(int) % n_bins
    return np.unique(b).size / float(n_bins)


def fit_circle_lsq(x, y):
    """Algebraic (Kasa) least-squares circle fit: x^2+y^2 + D x + E y + F = 0."""
    A = np.column_stack([x, y, np.ones_like(x)])
    sol, *_ = np.linalg.lstsq(A, -(x ** 2 + y ** 2), rcond=None)
    a, b = -sol[0] / 2.0, -sol[1] / 2.0
    r2 = a * a + b * b - sol[2]
    return (a, b, float(np.sqrt(r2))) if r2 > 0 else None


def edge_support(edges, x, y, r, tol=1.5):
    """Number of edge pixels within +-tol px of the circle (same yardstick for every method)."""
    ys, xs = np.nonzero(edges)
    return int((np.abs(np.hypot(xs - x, ys - y) - r) < tol).sum())


SAMPLE_RATIO = 0.8   # fraction of a circle's perimeter we expect Canny to keep as edge pixels


def suggest_params(edges, min_diam, max_diam, min_votes=None, confidence=0.5, verbose=True):
    """Turn the user's intent (diameter range, sensitivity) into detector parameters."""
    warnings = []
    n_edge = int(np.count_nonzero(edges))
    if min_diam <= 0 or max_diam <= 0:
        raise ValueError("MIN_DIAM and MAX_DIAM must be > 0")
    if min_diam > max_diam:
        warnings.append(f"MIN_DIAM ({min_diam}) > MAX_DIAM ({max_diam}) -> values swapped")
        min_diam, max_diam = max_diam, min_diam
    confidence = float(np.clip(confidence, 0.1, 1.0))

    r_min, r_max = 0.85 * min_diam / 2.0, 1.15 * max_diam / 2.0
    perim_min = 2.0 * np.pi * r_min

    if min_votes is None:                                    # automatic
        cht_votes = max(5, int(confidence * perim_min * SAMPLE_RATIO))
        votes_src = "auto  = confidence * 2*pi*r_min * SAMPLE_RATIO"
    else:                                                    # user override is respected
        cht_votes = int(min_votes)
        votes_src = "user-defined MIN_VOTES"

    ransac_min_inliers = cht_votes                           # same evidence level for both methods
    perim_frac = confidence * SAMPLE_RATIO                   # supporters must also cover this share of 2*pi*r (CHT + RANSAC)
    min_cov = float(np.clip(0.2 + 0.8 * confidence, 0.25, 0.9))   # share of the 360 deg that must be supported (0.6 at default)
    tol = 1.5 if n_edge < 1000 else (2.0 if n_edge < 3000 else 2.5)
    max_iter = int(np.clip(1.5 * n_edge, 1000, 8000))

    if n_edge < 100:
        warnings.append(f"only {n_edge} edge px (< 100): lower Canny thresholds / sigma")
    clutter = max(5000, int(0.03 * edges.shape[0] * edges.shape[1]))     # 5000 px, or 3% of the image if larger
    if n_edge > clutter:
        warnings.append(f"{n_edge} edge px (> {clutter}): cluttered - raise Canny thresholds or sigma, "
                        f"expect false circles and slow RANSAC")

    P = dict(min_diam=min_diam, max_diam=max_diam, r_min=r_min, r_max=r_max, n_edge=n_edge,
             CHT_MIN_VOTES=cht_votes, MIN_COVERAGE=min_cov,
             RANSAC_MIN_INLIERS=ransac_min_inliers, PERIM_FRAC=perim_frac,
             RANSAC_INLIER_TOL=tol, RANSAC_MAX_ITER=max_iter, warnings=warnings)
    if verbose:
        print("--- suggest_params ---------------------------------------")
        print(f" edge pixels          : {n_edge}")
        print(f" diameter range       : {min_diam} .. {max_diam} px  ->  r_min={r_min:.1f}, r_max={r_max:.1f}")
        print(f" CHT_MIN_VOTES        : {cht_votes}   ({votes_src})")
        print(f" PERIM_FRAC           : {perim_frac:.2f}  (CHT + RANSAC need >= this share of 2*pi*r as support)")
        print(f" MIN_COVERAGE         : {min_cov:.2f}  (angular coverage of the 360 deg)")
        print(f" RANSAC_MIN_INLIERS   : {ransac_min_inliers}   (same as CHT_MIN_VOTES)")
        print(f" RANSAC_INLIER_TOL    : {tol} px")
        print(f" RANSAC_MAX_ITER      : {max_iter}  (per extraction round)")
        for msg in warnings:
            print(" WARNING:", msg)
        print("----------------------------------------------------------")
    return P


def nms_circles(circles, center_frac=0.5, r_frac=0.25, min_center=4.0, min_dr=3.0):
    """Non-maximum suppression: drop a circle if a stronger one has (almost) the same
    centre AND (almost) the same radius. Concentric circles of different size are kept."""
    kept = []
    for c in sorted(circles, key=lambda c: -c["score"]):
        dup = False
        for k in kept:
            rm = min(c["r"], k["r"])
            if (np.hypot(c["x"] - k["x"], c["y"] - k["y"]) < max(min_center, center_frac * rm)
                    and abs(c["r"] - k["r"]) < max(min_dr, r_frac * rm)):
                dup = True
                break
        if not dup:
            kept.append(c)
    return kept


def circular_hough(gray_blur, edges, r_min, r_max, min_votes, min_coverage=0.5, perim_frac=0.0,
                   smooth=3, nms_win=7, align_cos=0.7, max_candidates=200, refine=True):
    """Circular Hough Transform from scratch (gradient-direction, 2-D centre accumulator).
    A circle needs max(min_votes, perim_frac * 2*pi*r) supporting edge pixels, so big circles need more votes.
    refine=True polishes each accepted circle with a least-squares fit on its supporting edge pixels (the accumulator
    centre can be several px off for large radii). Returns (circles, smoothed_accumulator);
    each circle = dict(x, y, r, score, coverage)."""
    H, W = edges.shape
    ys_i, xs_i = np.nonzero(edges)

    # 1) Sobel gradients -> unit normal vector at every edge pixel
    gx, gy = sobel_gradients(gray_blur.astype(np.float32))
    ex, ey = gx[ys_i, xs_i], gy[ys_i, xs_i]
    mag = np.hypot(ex, ey)
    ok = mag > 1e-6
    xs_i, ys_i, ex, ey, mag = xs_i[ok], ys_i[ok], ex[ok], ey[ok], mag[ok]
    if len(xs_i) < 3:
        return [], np.zeros((H, W), np.float32)
    nx, ny = ex / mag, ey / mag
    xs, ys = xs_i.astype(np.float64), ys_i.astype(np.float64)

    # 2) 2-D centre accumulator: vote along +-normal at every radius r_min..r_max
    acc = np.zeros(H * W, np.float64)
    radii = np.arange(int(np.floor(r_min)), int(np.ceil(r_max)) + 1)
    per_chunk = max(1, int(4_000_000 // (2 * len(xs))))     # radii per bincount call (keeps big images fast)
    for k in range(0, len(radii), per_chunk):
        votes = []
        for r in radii[k:k + per_chunk]:
            for s in (1.0, -1.0):                      # circle may be brighter or darker than background
                cx = np.rint(xs + s * r * nx).astype(np.int64)
                cy = np.rint(ys + s * r * ny).astype(np.int64)
                m = (cx >= 0) & (cx < W) & (cy >= 0) & (cy < H)
                votes.append(cy[m] * W + cx[m])
        acc += np.bincount(np.concatenate(votes), minlength=H * W)
    acc = acc.reshape(H, W).astype(np.float32)
    acc_s = conv2d_same(acc, np.ones((smooth, smooth), np.float32))   # absorb rounding errors

    # 3) local maxima of the accumulator = candidate centres
    is_peak = (acc_s >= max_filter(acc_s, nms_win)) & (acc_s >= 0.5 * min_votes)
    py, px = np.nonzero(is_peak)
    order = np.argsort(-acc_s[py, px])[:max_candidates]

    # 4) per candidate centre: radial-distance histogram -> supported radius,
    #    then perimeter support (vote count) + 360-degree angular coverage
    bins = np.arange(np.floor(r_min) - 0.5, np.ceil(r_max) + 1.5, 1.0)
    r_centers = 0.5 * (bins[:-1] + bins[1:])
    found = []
    for i in order:
        cx, cy = float(px[i]), float(py[i])
        dx, dy = xs - cx, ys - cy
        d = np.hypot(dx, dy)
        radial_ok = np.abs(dx * nx + dy * ny) / np.maximum(d, 1e-9) >= align_cos
        sel = (d >= r_min) & (d <= r_max) & radial_ok
        if sel.sum() < min_votes:
            continue
        ds = d[sel]
        hist, _ = np.histogram(ds, bins=bins)
        hs = np.convolve(hist, np.ones(3), mode="same")          # +-1 px smoothing
        for k in range(len(hs)):                                  # every radial peak (handles concentric circles)
            lo, hi = max(0, k - 2), min(len(hs), k + 3)
            if hs[k] < min_votes or hs[k] < hs[lo:hi].max() or (k > 0 and hs[k] == hs[k - 1]):
                continue
            r_est = float(ds[np.abs(ds - r_centers[k]) <= 1.5].mean())
            tol = 1.5 + 0.02 * r_est
            sup = sel & (np.abs(d - r_est) <= tol)
            ccx, ccy, rr = cx, cy, r_est
            if refine and sup.sum() >= 3:                       # least-squares polish of centre and radius
                for _ in range(2):
                    fit = fit_circle_lsq(xs[sup], ys[sup])
                    if fit is None or not (r_min <= fit[2] <= r_max):
                        break
                    ccx, ccy, rr = fit
                    dx, dy = xs - ccx, ys - ccy
                    d = np.hypot(dx, dy)
                    radial_ok = np.abs(dx * nx + dy * ny) / np.maximum(d, 1e-9) >= align_cos
                    sup = radial_ok & (np.abs(d - rr) <= 1.5 + 0.02 * rr)
            n_sup = int(sup.sum())
            cov = angular_coverage(dx[sup], dy[sup])
            if (n_sup >= max(min_votes, perim_frac * 2.0 * np.pi * rr)
                    and cov >= min_coverage and r_min <= rr <= r_max):
                found.append(dict(x=float(ccx), y=float(ccy), r=float(rr), score=n_sup, coverage=cov))

    # 5) NMS of duplicate detections
    return nms_circles(found), acc_s


def ransac_circles(edges, r_min, r_max, tol, max_iter, min_inliers, perim_frac=0.4,
                   min_coverage=0.5, batch=128, max_circles=30, refine=True, seed=0):
    """RANSAC circle detection from scratch.
    Each extraction round draws `max_iter` random 3-point samples, keeps the best circle,
    accepts it if it has enough support, removes its inliers and repeats."""
    rng = np.random.default_rng(seed)
    ys, xs = np.nonzero(edges)
    pts = np.column_stack([xs, ys]).astype(np.float64)
    circles = []

    for _ in range(max_circles):
        n = len(pts)
        if n < max(min_inliers, 3):
            break
        X, Y = pts[:, 0], pts[:, 1]
        best, drawn = None, 0
        while drawn < max_iter:
            B = min(batch, max_iter - drawn, max(8, int(4_000_000 // n)))     # cap memory for many edge points
            drawn += B
            idx = rng.integers(0, n, size=(B, 3))                         # 3-point sampling
            p1, p2, p3 = pts[idx[:, 0]], pts[idx[:, 1]], pts[idx[:, 2]]
            # circumcentre = intersection of two perpendicular bisectors:
            #   2(p2-p1).c = |p2|^2-|p1|^2   and   2(p3-p1).c = |p3|^2-|p1|^2
            a11, a12 = 2 * (p2[:, 0] - p1[:, 0]), 2 * (p2[:, 1] - p1[:, 1])
            a21, a22 = 2 * (p3[:, 0] - p1[:, 0]), 2 * (p3[:, 1] - p1[:, 1])
            b1 = (p2 ** 2).sum(1) - (p1 ** 2).sum(1)
            b2 = (p3 ** 2).sum(1) - (p1 ** 2).sum(1)
            det = a11 * a22 - a12 * a21
            ok = (np.abs(det) > 8.0) & (idx[:, 0] != idx[:, 1]) & (idx[:, 0] != idx[:, 2]) & (idx[:, 1] != idx[:, 2])
            det = np.where(ok, det, 1.0)                                  # non-collinear only
            ca = (b1 * a22 - a12 * b2) / det
            cb = (a11 * b2 - b1 * a21) / det
            cr = np.hypot(p1[:, 0] - ca, p1[:, 1] - cb)
            ok &= (cr >= r_min) & (cr <= r_max)                           # radius filter
            if not ok.any():
                continue
            ca, cb, cr = ca[ok], cb[ok], cr[ok]
            dist = np.abs(np.hypot(X[None, :] - ca[:, None], Y[None, :] - cb[:, None]) - cr[:, None])
            cnt = (dist < tol).sum(1)                                     # inlier scoring
            j = int(np.argmax(cnt))
            if best is None or cnt[j] > best[0]:
                best = (int(cnt[j]), ca[j], cb[j], cr[j])
        if best is None:
            break

        _, a, b, r = best
        mask = np.abs(np.hypot(X - a, Y - b) - r) < tol
        if refine and mask.sum() >= 3:                                    # least-squares polish on all inliers
            fit = fit_circle_lsq(X[mask], Y[mask])
            if fit is not None and r_min <= fit[2] <= r_max:
                a, b, r = fit
                mask = np.abs(np.hypot(X - a, Y - b) - r) < tol
        n_in = int(mask.sum())
        cov = angular_coverage(X[mask] - a, Y[mask] - b)
        if n_in < max(min_inliers, perim_frac * 2 * np.pi * r) or cov < min_coverage:
            break                                                         # best of this round is not a circle -> done
        circles.append(dict(x=float(a), y=float(b), r=float(r), score=n_in, coverage=float(cov)))
        pts = pts[~mask]                                                  # remove inliers, extract next circle
    return circles


def _to_gray_uint8(src):
    """bytes (png/jpg) or ndarray (gray / BGR / BGRA, uint8 or float) -> 2-D uint8."""
    if isinstance(src, (bytes, bytearray)):
        img = cv2.imdecode(np.frombuffer(src, np.uint8), cv2.IMREAD_UNCHANGED)
        if img is None:
            raise ValueError("could not decode image bytes")
    else:
        img = np.asarray(src)
    if img.dtype != np.uint8:
        img = img.astype(np.float64)
        if img.max() <= 1.0:
            img = img * 255.0
        img = np.clip(img, 0, 255).astype(np.uint8)
    if img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY)
        elif img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            img = img[..., 0]
    if img.ndim != 2:
        raise ValueError("unsupported image shape " + str(img.shape))
    return np.ascontiguousarray(img)


def _auto_high_threshold(grad):
    """Automatic high threshold for the scratch Canny, independent of the image size: 5x the median
    Sobel gradient magnitude (the median is dominated by flat/noisy background, so it measures the noise
    level), but never below 15% of the 99.5th percentile (keeps noise-free images sensible)."""
    return float(max(5.0 * np.median(grad), 0.15 * np.percentile(grad, 99.5), 1.0))


def detect_circles_pipeline(image_array_or_bytes, method="cht", min_diam=10, max_diam=100,
                            min_votes=None, confidence=0.5, sigma=1.5,
                            canny_low=None, canny_high=None, seed=0,
                            canny_engine="scratch", canny_kernel=5):
    """End-to-end circle detection (no notebook globals). Returns a JSON-serialisable dict.
    The image is processed at its original size (no resizing): min_diam / max_diam, centres and radii are
    all in image pixels. canny_engine="scratch" uses the from-scratch Canny functions in this module
    (thresholds on the Sobel gradient magnitude; automatic noise-based values when None);
    "opencv" uses cv2.Canny (much faster)."""
    t0 = time.perf_counter()
    try:
        method = str(method).lower()
        if method not in ("cht", "ransac"):
            raise ValueError("method must be 'cht' or 'ransac'")
        if canny_engine not in ("scratch", "opencv"):
            raise ValueError("canny_engine must be 'scratch' or 'opencv'")
        gray = _to_gray_uint8(image_array_or_bytes)
        h0, w0 = gray.shape
        small = gray
        mp = h0 * w0 / 1e6
        size_warning = []
        if canny_engine == "scratch" and mp > 1.0:
            size_warning.append(f"{mp:.1f} MP image: the pure-Python Canny needs roughly "
                                f"{1.33 * int(canny_kernel) * mp:.0f} s; use canny_engine='opencv' or a smaller image if too slow")
        if canny_engine == "scratch":
            blur_f = gaussian_blur(small, int(canny_kernel), float(sigma))
            grad, ang = apply_sobel(blur_f)
            nms = non_maximum_suppression(grad, ang)
            if canny_high is None:
                canny_high = _auto_high_threshold(grad)
            if canny_low is None:
                canny_low = 0.5 * canny_high
            validate_parameters(int(canny_kernel), float(sigma), float(canny_high), float(canny_low))
            edges = (hysteresis_thresholding(nms, float(canny_high), float(canny_low)) > 0).astype(np.uint8) * 255
            blur = np.clip(np.rint(blur_f), 0, 255).astype(np.uint8)
        else:
            blur = cv2.GaussianBlur(small, (0, 0), float(sigma))
            if canny_high is None:
                otsu, _ = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                canny_high = max(30.0, float(otsu))
            if canny_low is None:
                canny_low = 0.5 * canny_high
            edges = cv2.Canny(blur, float(canny_low), float(canny_high))

        P = suggest_params(edges, min_diam, max_diam, min_votes, confidence, verbose=False)

        if method == "cht":
            found, _ = circular_hough(blur, edges, P["r_min"], P["r_max"], P["CHT_MIN_VOTES"], P["MIN_COVERAGE"], P["PERIM_FRAC"])
        else:
            found = ransac_circles(edges, P["r_min"], P["r_max"], P["RANSAC_INLIER_TOL"], P["RANSAC_MAX_ITER"],
                                   P["RANSAC_MIN_INLIERS"], P["PERIM_FRAC"], P["MIN_COVERAGE"], seed=seed)

        circles = []
        for c in sorted(found, key=lambda c: -c["score"]):
            score = c["coverage"]                                      # share of the 360 deg with edge support
            circles.append({
                "center": [round(float(c["x"]), 1), round(float(c["y"]), 1)],
                "radius": round(float(c["r"]), 1),
                "score": round(float(score), 2),
                "original_coords": {"center": [round(float(c["x"]), 1), round(float(c["y"]), 1)],
                                    "radius": round(float(c["r"]), 1)}})
        return {"status": "success", "method_used": method, "count": len(circles), "circles": circles,
                "warnings": size_warning + list(P["warnings"]),
                "execution_time_ms": round((time.perf_counter() - t0) * 1000.0, 1)}
    except Exception as e:
        return {"status": "error", "message": str(e), "method_used": str(method), "count": 0, "circles": [],
                "execution_time_ms": round((time.perf_counter() - t0) * 1000.0, 1)}

# ---- quick self-test of the exportable function ----
ok, buf = cv2.imencode(".png", gray_full)
for m in ("cht", "ransac"):
    out = detect_circles_pipeline(buf.tobytes(), method=m, min_diam=MIN_DIAM, max_diam=MAX_DIAM)
    print(json.dumps({k: (v if k != "circles" else v[:2]) for k, v in out.items()}, indent=2))
print("Tip: copy this cell into circle_detector.py to import it from a Flask / FastAPI backend.")